In [2]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Attention from scratch
Goal: understand self-attention by solving the "first digit" memory task
that RNN / LSTM / CNN struggled with.

In [3]:
def make_memory_data(n, length=30):
    """Random digits 0-9; label is 1 if the FIRST digit is > 4."""
    x = torch.randint(0, 10, (n, length))   # (n, length)
    y = (x[:, 0] > 4).long()                # (n,)
    return x, y

In [4]:
x, y = make_memory_data(5)
print(x.shape, y.shape)
print(x[:, 0])
print(y)

torch.Size([5, 30]) torch.Size([5])
tensor([2, 2, 6, 7, 9])
tensor([0, 0, 1, 1, 1])


## A2. Attention by hand
Each word compares itself with every word (dot product),
softmax turns scores into weights (sum = 1),
the new vector is a weighted average of all vectors.

In [5]:
def toy_attention(x, idx):
    """Attention for ONE word (row idx of x) over all words in x."""
    scores = x @ x[idx]                      # similarity with every word, shape (n_words,)
    weights = torch.softmax(scores, dim=0)   # weights sum to 1
    new_vec = weights @ x                    # weighted average of all vectors
    return scores, weights, new_vec

# toy vectors: [how "alive", how "place/object"]
words = ["kot", "ulice", "byl"]
x_toy = torch.tensor([[1.0, 0.1],    # kot
                      [0.1, 1.0],    # ulice
                      [0.9, 0.2]])   # byl: looks for something alive

scores, weights, new_vec = toy_attention(x_toy, idx=2)
print("scores :", scores)
print("weights:", weights, "sum =", weights.sum().item())
print("new byl:", new_vec)

scores : tensor([0.9200, 0.2900, 0.8500])
weights: tensor([0.4057, 0.2161, 0.3783]) sum = 1.0
new byl: tensor([0.7677, 0.3323])


## A3. Self-attention with learned Q, K, V
Each word is projected three times (Linear layers):
- Q (query): what am I looking for?
- K (key): what do I offer as a label?
- V (value): what do I pass on if someone attends to me?
scores = Q K^T / sqrt(d) -> softmax -> weighted average of V.

In [6]:
def self_attention(x, w_q, w_k, w_v):
    """Single-head self-attention. x has shape (n_words, d)."""
    q = w_q(x)                          # (n_words, d) what each word is looking for
    k = w_k(x)                          # (n_words, d) label each word offers
    v = w_v(x)                          # (n_words, d) content each word passes on
    d = q.shape[-1]
    scores = q @ k.T / math.sqrt(d)     # (n_words, n_words) everyone vs everyone
    weights = torch.softmax(scores, dim=-1)   # each ROW sums to 1
    return weights @ v, weights         # new vectors (n_words, d) and the weights


torch.manual_seed(0)
x = torch.randn(3, 4)                   # 3 words, 4 numbers each (random for now)
w_q, w_k, w_v = (nn.Linear(4, 4, bias=False) for _ in range(3))

out, weights = self_attention(x, w_q, w_k, w_v)
print("out shape    :", out.shape)
print("weights shape:", weights.shape)
print(weights)
print("row sums     :", weights.sum(dim=-1))

out shape    : torch.Size([3, 4])
weights shape: torch.Size([3, 3])
tensor([[0.3484, 0.3430, 0.3087],
        [0.1911, 0.4861, 0.3228],
        [0.2104, 0.4680, 0.3216]], grad_fn=<SoftmaxBackward0>)
row sums     : tensor([1., 1., 1.], grad_fn=<SumBackward1>)


## A4. Attention on the "first digit" task
Embedding + position -> self-attention -> read the LAST token -> Linear.
If it works, the last token must learn to look at position 0.

In [7]:
class AttentionNet(nn.Module):
    """Embedding (+ position) -> one self-attention layer -> classify from LAST token."""

    def __init__(self, vocab_size=10, d=32, length=10, use_pos=True):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d)
        self.pos = nn.Embedding(length, d) if use_pos else None   # which position is it?
        self.w_q = nn.Linear(d, d, bias=False)
        self.w_k = nn.Linear(d, d, bias=False)
        self.w_v = nn.Linear(d, d, bias=False)
        self.fc = nn.Linear(d, 2)
        self.last_weights = None                                   # saved for inspection

    def forward(self, x):                                          # x: (batch, length)
        h = self.emb(x)                                            # (batch, length, d)
        if self.pos is not None:
            h = h + self.pos(torch.arange(x.shape[1], device=x.device))
        q, k, v = self.w_q(h), self.w_k(h), self.w_v(h)            # (batch, length, d)
        scores = q @ k.transpose(1, 2) / math.sqrt(q.shape[-1])    # (batch, length, length)
        weights = torch.softmax(scores, dim=-1)
        self.last_weights = weights.detach()
        out = weights @ v                                          # (batch, length, d)
        return self.fc(out[:, -1])                                 # only the last token


def make_loaders(n_train=20000, n_val=2000, length=10, batch_size=64):
    x_tr, y_tr = make_memory_data(n_train, length)
    x_va, y_va = make_memory_data(n_val, length)
    train_loader = DataLoader(TensorDataset(x_tr, y_tr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(x_va, y_va), batch_size=batch_size)
    return train_loader, val_loader


def run_epoch(model, loader, optimizer=None):
    """One pass over the data; trains if an optimizer is given. Returns (loss, acc)."""
    training = optimizer is not None
    model.train(training)
    loss_fn = nn.CrossEntropyLoss()
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            n += len(yb)
    return total_loss / n, correct / n


def train_attention(use_pos=True, length=10, epochs=10, lr=3e-3):
    torch.manual_seed(42)
    train_loader, val_loader = make_loaders(length=length)
    model = AttentionNet(length=length, use_pos=use_pos).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, optimizer)
        va_loss, va_acc = run_epoch(model, val_loader)
        print(f"Epoch {epoch}/{epochs} | train acc {tr_acc:.3f} | val loss {va_loss:.3f} acc {va_acc:.3f}")
    return model, val_loader


model, val_loader = train_attention(use_pos=True)

# Where does the LAST token look? (weights of the last row, first validation sample)
xb, yb = next(iter(val_loader))
model.eval()
model(xb.to(device))
torch.set_printoptions(precision=2, sci_mode=False)
print(xb[0])
print(model.last_weights[0, -1])

Epoch 1/10 | train acc 0.971 | val loss 0.000 acc 1.000
Epoch 2/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 3/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 4/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 5/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 6/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 7/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 8/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 9/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 10/10 | train acc 1.000 | val loss 0.000 acc 1.000
tensor([7, 1, 3, 0, 6, 2, 9, 1, 2, 1])
tensor([1.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00])


## A5. Positional embedding: why attention needs it

Self-attention treats the input as a *set*: it has no idea which digit came first.
Fix: add a learned vector per position, `h = emb(digit) + pos(position)`.

**Experiment:** same model, same data (length 10), with and without `pos`.

| model | val acc |
|---|---|
| attention + pos | 1.000 |
| attention, no pos | ~0.62 |

Why ~0.62 and not 0.5? Without order the model still sees *how many* digits are > 4.
The first digit is one of them, so more big digits make "first digit is big" more likely.

In [8]:
model_no_pos, _ = train_attention(use_pos=False)

Epoch 1/10 | train acc 0.615 | val loss 0.650 acc 0.608
Epoch 2/10 | train acc 0.628 | val loss 0.640 acc 0.614
Epoch 3/10 | train acc 0.630 | val loss 0.638 acc 0.623
Epoch 4/10 | train acc 0.631 | val loss 0.638 acc 0.624
Epoch 5/10 | train acc 0.629 | val loss 0.639 acc 0.619
Epoch 6/10 | train acc 0.632 | val loss 0.637 acc 0.619
Epoch 7/10 | train acc 0.631 | val loss 0.638 acc 0.622
Epoch 8/10 | train acc 0.629 | val loss 0.639 acc 0.618
Epoch 9/10 | train acc 0.631 | val loss 0.634 acc 0.622
Epoch 10/10 | train acc 0.632 | val loss 0.638 acc 0.626


In [9]:
class BaselineNet(nn.Module):
    """Embedding -> RNN / LSTM / CNN -> Linear(2). Same task and data as AttentionNet."""

    def __init__(self, kind, vocab_size=10, d=32):
        super().__init__()
        self.kind = kind
        self.emb = nn.Embedding(vocab_size, d)
        if kind == "rnn":
            self.core = nn.RNN(d, d, batch_first=True)
        elif kind == "lstm":
            self.core = nn.LSTM(d, d, batch_first=True)
        elif kind == "cnn":
            self.core = nn.Conv1d(d, d, kernel_size=3, padding=1)
        else:
            raise ValueError(kind)
        self.fc = nn.Linear(d, 2)

    def forward(self, x):                                  # x: (batch, length)
        h = self.emb(x)                                    # (batch, length, d)
        if self.kind == "cnn":
            h = torch.relu(self.core(h.transpose(1, 2)))   # Conv1d wants (batch, d, length)
            return self.fc(h.max(dim=2).values)            # global max pooling over positions
        out, _ = self.core(h)                              # (batch, length, d)
        return self.fc(out[:, -1])                         # last time step only


def set_forget_bias(lstm, value=2.0):
    """Set the forget-gate bias. PyTorch bias layout is [input | forget | cell | output]."""
    d = lstm.hidden_size
    with torch.no_grad():
        lstm.bias_ih_l0[d:2 * d].fill_(value)


def train_model(model, length, epochs=10, lr=3e-3, n_train=50000):
    """Train any of our models and return the best validation accuracy."""
    train_loader, val_loader = make_loaders(n_train=n_train, length=length)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_acc = 0.0
    for _ in range(epochs):
        run_epoch(model, train_loader, optimizer)
        _, val_acc = run_epoch(model, val_loader)
        best_acc = max(best_acc, val_acc)
    return best_acc


def compare_models(length=30, epochs=10):
    """Train every model on the same task and print the best val accuracy of each."""
    results = {}
    for name in ("rnn", "lstm", "lstm_bias", "cnn", "attention"):
        torch.manual_seed(42)
        if name == "attention":
            model = AttentionNet(length=length)
        else:
            model = BaselineNet("lstm" if name == "lstm_bias" else name)
        if name == "lstm_bias":
            set_forget_bias(model.core, 2.0)
        results[name] = train_model(model, length, epochs)
        print(f"{name:10s} best val acc: {results[name]:.3f}")
    return results


results_30 = compare_models(length=30)

rnn        best val acc: 1.000
lstm       best val acc: 0.512
lstm_bias  best val acc: 1.000
cnn        best val acc: 1.000
attention  best val acc: 1.000


no padding

In [10]:
torch.manual_seed(42)
cnn_valid = BaselineNet("cnn")
cnn_valid.core = nn.Conv1d(32, 32, kernel_size=3, padding=0)   # no padding at the edges
print(train_model(cnn_valid, length=30))

0.6275


## A6. Comparison: RNN vs LSTM vs CNN vs attention

**Task:** random digits 0-9, label = 1 if the FIRST digit is > 4.
The model must remember one early digit until the end of the sequence.

| model | length 10 (val acc) | length 30 (val acc) |
|---|---|---|
| RNN | 1.00 | 1.00 |
| LSTM | 1.00 | 0.50 (stuck) |
| CNN (Conv1D + global max pooling) | 0.80 | 0.63 |
| Attention + pos | 1.00 | 1.00 |

(Keras runs; the length-30 attention result is filled in after running A6.)

**Why each one behaves this way**
- **RNN:** with enough data it can carry the first digit through all steps,
  but the gradient shrinks at every step (vanishing gradient), so it is unstable on long sequences.
- **LSTM:** the forget gate starts around 0.5, so after 29 steps the signal is
  about 0.5**29 ≈ 2e-9. Initializing the forget-gate bias to a positive value
  (e.g. 2.0) fixed it (val acc 1.000).
- **CNN:** global max pooling keeps only the strongest value per filter and drops
  positions, so it cannot say "the first digit".
- **Attention:** the last token looks directly at position 0 (weight ≈ 1.00),
  so distance does not matter. It needs a positional embedding, otherwise
  val acc drops to ~0.62.

**Trade-off:** attention builds an n × n weight table, so memory and compute
grow quadratically with sequence length.

In [11]:
model30, _ = train_attention(use_pos=True, length=30)

Epoch 1/10 | train acc 0.988 | val loss 0.000 acc 1.000
Epoch 2/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 3/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 4/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 5/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 6/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 7/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 8/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 9/10 | train acc 1.000 | val loss 0.000 acc 1.000
Epoch 10/10 | train acc 1.000 | val loss 0.000 acc 1.000


In [12]:
def show_last_token_scores(model, x):
    """Print the last token's score against every position (before softmax)."""
    model.eval()
    with torch.no_grad():
        h = model.emb(x) + model.pos(torch.arange(x.shape[1], device=x.device))
        q, k = model.w_q(h), model.w_k(h)
        scores = q @ k.transpose(1, 2) / math.sqrt(q.shape[-1])
    return scores[0, -1]          # last token vs each of the 10 positions


xb, yb = next(iter(val_loader))
print(xb[0])
print(show_last_token_scores(model, xb.to(device)))

tensor([7, 1, 3, 0, 6, 2, 9, 1, 2, 1])
tensor([11.30, -5.16, -3.14, -1.38,  3.48, -5.30, -0.73, -5.48, -0.73, -1.45])


# Part B: a small transformer (character-level GPT) on Pan Tadeusz

Goal: train a small model on the text of "Pan Tadeusz" (Adam Mickiewicz, public domain)
that predicts the next character, then generate new Polish-sounding text from it.

Pipeline:
text -> tokenizer -> IDs -> batches -> embedding (+ position)
-> [ causal self-attention -> feed-forward ] x N -> Linear -> next-character probabilities

| step | what we add |
|---|---|
| B1 | character tokenizer + data |
| B2 | batches (input and shifted target) |
| B3 | baseline model (no attention) |
| B4 | causal mask |
| B5 | multi-head attention |
| B6 | transformer block (feed-forward, residual, layer norm) |
| B7 | stacked blocks, training, text generation |

Why a different text than the video we watch: same steps, new data, so we can
check that we understand the concept and not just the code.

## B1. Character tokenizer and data

Token = one character. Vocabulary = all distinct characters in the text
(here it also includes Polish letters like ą, ę, ł, ó, ś, ż).

- Clean the raw file: the Wolne Lektury file contains a title header and a license/footer
  that are not part of the poem, so we cut them off.
- `stoi`: character -> ID, `itos`: ID -> character.
- `encode(text)` -> list of IDs, `decode(ids)` -> text.
- The whole text becomes one long tensor of IDs.
- Split: first 90% train, last 10% validation (text the model never trained on).

Vocabulary size: 445321 (fill in after running).

Why characters and not BPE: the vocabulary is tiny and there is nothing to train,
so we can focus on the transformer. The price is longer sequences.

In [13]:
import os
import urllib.request

URL = "https://wolnelektury.pl/media/book/txt/pan-tadeusz.txt"
PATH = "pan_tadeusz.txt"


def load_text(url=URL, path=PATH):
    """Download the text once (if missing) and return it as one string."""
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


text = load_text()
print("characters:", len(text))


characters: 447334


Clean the text 

In [14]:
def clean_text(raw):
    """Keep only the poem: cut the header before 'Litwo!' and the footer after '-----'."""
    start = raw.find("Litwo! Ojczyzno moja!")
    end = raw.rfind("-----")            # last separator line = start of the footer
    return raw[start:end].strip()

text = clean_text(load_text())
print("characters:", len(text))
print(repr(text[:60]))
print(repr(text[-60:]))

characters: 445321
'Litwo! Ojczyzno moja! ty jesteś jak zdrowie;\nIle cię trzeba '
'kami wieśniaczki usnuty\nZ modrych bławatków i zielonej ruty…'


Build dictionary of words

In [15]:
def build_vocab(text):
    """Return the sorted list of distinct characters and the two lookup dicts."""
    chars = sorted(set(text))                      # distinct characters, fixed order
    stoi = {ch: i for i, ch in enumerate(chars)}   # character -> ID
    itos = {i: ch for i, ch in enumerate(chars)}   # ID -> character
    return chars, stoi, itos


def encode(s, stoi):
    """Text -> list of IDs."""
    return [stoi[ch] for ch in s]


def decode(ids, itos):
    """List of IDs -> text."""
    return "".join(itos[i] for i in ids)

chars, stoi, itos = build_vocab(text)
ids = encode("Adam", stoi)
print(ids)
print(decode(ids, itos))
print(stoi)

[16, 42, 39, 51]
Adam
{'\n': 0, ' ': 1, '!': 2, '(': 3, ')': 4, '*': 5, ',': 6, '-': 7, '.': 8, '/': 9, '1': 10, '2': 11, '8': 12, ':': 13, ';': 14, '?': 15, 'A': 16, 'B': 17, 'C': 18, 'D': 19, 'E': 20, 'F': 21, 'G': 22, 'H': 23, 'I': 24, 'J': 25, 'K': 26, 'L': 27, 'M': 28, 'N': 29, 'O': 30, 'P': 31, 'R': 32, 'S': 33, 'T': 34, 'U': 35, 'V': 36, 'W': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64, '«': 65, '»': 66, 'Ó': 67, 'à': 68, 'æ': 69, 'é': 70, 'ó': 71, 'ą': 72, 'Ć': 73, 'ć': 74, 'ę': 75, 'Ł': 76, 'ł': 77, 'ń': 78, 'Ś': 79, 'ś': 80, 'Ź': 81, 'ź': 82, 'Ż': 83, 'ż': 84, '—': 85, '…': 86}


In [16]:
import torch


def make_splits(text, stoi, train_frac=0.9):
    """Encode the whole text into one tensor and split it into train / val."""
    data = torch.tensor(encode(text, stoi), dtype=torch.long)  # shape: (n_chars,)
    n_train = int(train_frac * len(data))                       # first 90% of positions
    return data, data[:n_train], data[n_train:]


data, train_data, val_data = make_splits(text, stoi)
print(data.shape, train_data.shape, val_data.shape)
print(data[:15])
print(decode(data[:15].tolist(), itos))

torch.Size([445321]) torch.Size([400788]) torch.Size([44533])
tensor([27, 47, 58, 61, 53,  2,  1, 30, 48, 41, 64, 63, 64, 52, 53])
Litwo! Ojczyzno


## B2. Batches

The model sees a window of `block_size` characters (the context) and predicts the next one.

- `x` = a random chunk of length `block_size`, shape `(batch, block_size)`.
- `y` = the same chunk shifted by one character (the targets).
- One chunk gives `block_size` training examples at once: at position t the input is
  characters 0..t and the target is character t+1.
- `get_batch(split)` picks random start positions from train or validation data.

Shapes: `x` and `y` are both `(batch, block_size)`.

In [17]:
def get_batch(data, block_size, batch_size):
    """Return a random batch: x (chunks) and y (the same chunks shifted by one char)."""
    # random start positions; must leave room for block_size + 1 chars
    starts = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in starts])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in starts])
    return x, y

xb, yb = get_batch(train_data, block_size=8, batch_size=4)
print(xb.shape, yb.shape)
print(xb)
print(yb)

torch.Size([4, 8]) torch.Size([4, 8])
tensor([[51, 71, 45, 77,  1, 57, 47, 75],
        [41, 47, 43, 80, 52, 47, 53, 52],
        [47, 43, 52, 47, 77, 63,  1, 48],
        [ 1,  1, 25, 59, 84,  1, 51, 47]])
tensor([[71, 45, 77,  1, 57, 47, 75,  1],
        [47, 43, 80, 52, 47, 53, 52, 63],
        [43, 52, 47, 77, 63,  1, 48, 72],
        [ 1, 25, 59, 84,  1, 51, 47, 43]])


how it works?

In [18]:
print(repr(decode(xb[2].tolist(), itos)))
print(repr(decode(yb[2].tolist(), itos)))

'ieniły j'
'eniły ją'


## B3. Baseline: bigram model

No attention yet. Each character looks only at itself: an embedding table of shape
`(vocab_size, vocab_size)` gives the scores (logits) for the next character directly.

- Loss: cross-entropy between logits and `y`.
- Sanity check: before training the loss should be about ln(vocab_size) (pure guessing).
- `generate(...)`: take the logits of the last position, softmax, sample one character,
  append it, repeat.

Baseline validation loss after training: __ (fill in).
Every later step should lower it, and the generated text should look more like Polish.

one initialization

In [19]:
import torch.nn as nn
import torch.nn.functional as F


class BigramModel(nn.Module):
    """Predict the next char from the current char only (a lookup table)."""

    def __init__(self, vocab_size):
        super().__init__()
        # row = current char, columns = scores for every possible next char
        self.table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, x, y=None):
        logits = self.table(x)                       # (batch, block, vocab)
        if y is None:
            return logits, None
        b, t, v = logits.shape
        loss = F.cross_entropy(logits.view(b * t, v), y.view(b * t))
        return logits, loss


vocab_size = len(chars)
model = BigramModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss.item())

torch.Size([4, 8, 87])
4.872867107391357


more initializations

In [20]:
def train_bigram(model, train_data, steps, block_size, batch_size, lr=1e-2):
    """Train the model for a number of steps; print the loss every 500 steps."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for step in range(steps):
        xb, yb = get_batch(train_data, block_size, batch_size)  # new random batch
        _, loss = model(xb, yb)                                 # guess + loss
        optimizer.zero_grad()                                   # clear old corrections
        loss.backward()                                         # which cells go up / down
        optimizer.step()                                        # apply small corrections
        if step % 500 == 0:
            print(step, loss.item())
    return model


model = BigramModel(vocab_size)
train_bigram(model, train_data, steps=3000, block_size=8, batch_size=32)

0 5.010759353637695
500 2.7277748584747314
1000 2.5890049934387207
1500 2.604912519454956
2000 2.594587564468384
2500 2.591883420944214


BigramModel(
  (table): Embedding(87, 87)
)

In [21]:
@torch.no_grad()   # no learning here, just writing
def generate(model, max_new_tokens, start_id=0):
    """Generate text one char at a time by sampling from the model's probabilities."""
    idx = torch.tensor([[start_id]], dtype=torch.long)       # shape (1, 1)
    for _ in range(max_new_tokens):
        logits, _ = model(idx)
        probs = F.softmax(logits[:, -1, :], dim=-1)           # scores -> percentages
        next_id = torch.multinomial(probs, num_samples=1)     # draw one char by chance
        idx = torch.cat([idx, next_id], dim=1)                # append and repeat
    return idx[0].tolist()


print(decode(generate(model, 300), itos))


 rakro orwile  krd, poństrażo d wa kieczyszani, by zy nalspszasiczą pstustra, si sta ż w: wakwi i
Pci Hrzecnałużew wi!» wych grólele gwśle, drcznied ito tóranaromu).
I te jały prsem pubiówczę, ie. tka Wiej dłą; ws n j tych;
Lew nesiąw, nie ki
 nienagł
Dy dy.
Najsachalkru,
Ra pa mosie zyneje wy!
Ch n


before and after training

In [22]:
untrained = BigramModel(vocab_size)
print(decode(generate(untrained, 200), itos))   # before training
print("-----")
print(decode(generate(model, 200), itos))       # after training


śV;a
8c«NAPe»ààvOTxĆdVu:-ńruŹæuV1ŚTæSęoASOO,—hŻgGFlŁŁwOfjz«EkévK,:Uą/—*ÓJ,ŚDęŻ:WKeq.wIćÓi/-E»àĆxtRc.VŻfO2ćDVłź1!:J,łT. (ĆhpŹ2ŻJ źræÓcfC18-ńjÓ;w»DjŹ!«F)x2àŹp«-ŻNT?cŻ*
źnIęZ!*LszćapllłæńK…v
æuà
…J,)AŹżI
-----

Mo iazyj pieli Ho wę i otem  cici wichonie otej bu: zę;  dzu;
 wsach, kłysiczbiewi zegdz, d zlakłuch Takodzcabę dz kozę,
I nasznogów w owięszymch j pijerysnie pli nimieschuck bry ! wspy n siadzgla rad


## B4. Causal mask

Self-attention lets every token look at every other token. For next-character
prediction that is cheating: token t would see the character it has to predict.

Fix: token t may attend only to positions <= t.

- Build a lower-triangular matrix of ones, shape `(T, T)`.
- Before softmax, set the scores of the future positions to `-inf`.
- After softmax those positions get weight 0, and every row still sums to 1.

Check: print the weights of one sequence, the upper triangle must be all zeros.

In [23]:
def average_previous(x):
    """For each position, average the vectors of all positions up to and including it."""
    t = x.shape[-2]                                   # number of positions
    mask = torch.tril(torch.ones(t, t))               # lower triangle of ones
    weights = mask / mask.sum(dim=1, keepdim=True)    # every row sums to 1
    return weights @ x                                # works for (t, c) and (b, t, c)


def causal_weights(t):
    """Attention weights with equal scores, but future positions masked out."""
    scores = torch.zeros(t, t)                         # same score for every pair
    mask = torch.tril(torch.ones(t, t))                # 1 = allowed, 0 = future
    scores = scores.masked_fill(mask == 0, float("-inf"))
    return F.softmax(scores, dim=-1)                   # each row -> percentages


print(causal_weights(3))

tensor([[1.00, 0.00, 0.00],
        [0.50, 0.50, 0.00],
        [0.33, 0.33, 0.33]])


## B5. Multi-head attention

One attention head can focus on one kind of pattern. Several heads in parallel can
look for different things at the same time.

- Split the vector size `d` into `n_heads` parts of size `d / n_heads`.
- Each head has its own Q, K, V and its own attention weights.
- Concatenate the head outputs back to size `d` and pass them through one Linear layer.

Shapes: input `(batch, T, d)` -> each head `(batch, T, d / n_heads)` -> concat `(batch, T, d)`.

## B6. Transformer block

One block = attention + feed-forward, wrapped with two stabilizers.

- Feed-forward: Linear(d -> 4d), non-linearity, Linear(4d -> d), applied to each token separately.
  Attention mixes information between tokens, feed-forward processes each token on its own.
- Residual connection: `x = x + sublayer(x)`. The input is added back, so gradients
  and the original signal have a shortcut.
- Layer norm: normalizes each token vector, applied before attention and before feed-forward.
- Dropout: randomly zeroes some values during training to reduce overfitting.

Shape does not change: `(batch, T, d)` in, `(batch, T, d)` out.

## B7. Full model, training, generation

Model:
token embedding + position embedding -> N blocks -> final layer norm -> Linear(d -> vocab_size)

Training: AdamW, cross-entropy on `(logits, y)`, check train and validation loss every N steps.

| model | validation loss |
|---|---|
| bigram baseline (B3) | __ |
| + attention (B4) | __ |
| + multi-head, block, N blocks (B7) | __ |

Generation: start from one character (or a prompt), repeat:
1. crop the context to the last `block_size` characters,
2. run the model, take the logits of the last position,
3. softmax, sample a character, append it.

Note: the model sees at most `block_size` characters back, because the position
embedding only has that many rows.